In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
# os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1" # Enable in M1 Mac CPUs

from matplotlib import rcParams
import matplotlib.pyplot as plt


# Some preambles for prettification
rcParams.update({'figure.figsize': (8, 6), 'axes.spines.top': False,
                 'axes.spines.right': False, 'axes.labelsize': 12,
                 'axes.titlesize': 12, 'axes.titleweight': 'bold',
                 'lines.linewidth': 1.5})

Reference: Chollet, F., & Watson, M. (2026). Deep learning with Python Third Edition. Manning.

Prepared by: Leodegario Lorenzo II

# Natural Language Processing using RNNs

In [ ]:
import keras
from keras import layers

## 1 Data Preparation

### Downloading the Dataset

In [ ]:
import os, pathlib, shutil, random

In [ ]:
# zip_path = keras.utils.get_file(
#     origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
#     extract=True,
# )

# imdb_extract_dir = pathlib.Path(zip_path) / "aclImdb"

In [ ]:
# shutil.copytree(imdb_extract_dir, 'data/aclImdb');

In [ ]:
data_dir = pathlib.Path('data/aclImdb')

### Sample Data

In [ ]:
with open(data_dir / "train" / "pos" / "4077_10.txt") as f:
    print(f.read())

In [ ]:
from glob import glob

import numpy as np

In [ ]:
np.random.seed(1337)
with open(np.random.choice(glob(f"{str(data_dir)}/train/neg/*.txt"))) as f:
    print(f.read())

### Data Segregation

In [ ]:
train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

Copy files to test directory

In [ ]:
# shutil.copytree(data_dir / "test", test_dir);

Create a train-validation test data by randomly selecting data in our training set

In [ ]:
# val_percentage = 0.2

# for category in ("neg", "pos"):
#     src_dir = data_dir / "train" / category
#     src_files = os.listdir(src_dir)
#     random.Random(1337).shuffle(src_files)
#     num_val_samples = int(len(src_files) * val_percentage)

#     os.makedirs(val_dir / category)
#     for file in src_files[:num_val_samples]:
#         shutil.copy(src_dir / file, val_dir / category / file)
#     os.makedirs(train_dir / category)
#     for file in src_files[num_val_samples:]:
#         shutil.copy(src_dir / file, train_dir / category / file)

### Creation of the Dataset

In [ ]:
from keras.utils import text_dataset_from_directory

In [ ]:
batch_size = 8

train_ds = text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds = text_dataset_from_directory(val_dir, batch_size=batch_size)
test_ds = text_dataset_from_directory(test_dir, batch_size=batch_size)

In [ ]:
train_ds_no_labels = train_ds.map(lambda x, y: x)

## 2 Set-based Machine Learning Model

In [ ]:
from keras import layers

### Bag-of-Words Model

#### Text Vectorization

In [ ]:
max_tokens = 20_000

text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="multi_hot",
)

In [ ]:
text_vectorization.adapt(train_ds_no_labels)

In [ ]:
bag_of_words_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bag_of_words_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bag_of_words_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)

In [ ]:
x, y = next(bag_of_words_train_ds.as_numpy_iterator())
x.shape

In [ ]:
y.shape

#### Model Building

In [ ]:
max_tokens = 20_000
name = "bag_of_words_classifier"

inputs = keras.Input(shape=(max_tokens,))
outputs = layers.Dense(1, activation="sigmoid")(inputs)
model = keras.Model(inputs, outputs, name=name)

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=2,
)
model_checkpoint = keras.callbacks.ModelCheckpoint(
    "models/bow_model.keras", save_best_only=True
)

In [ ]:
history = model.fit(
    bag_of_words_train_ds,
    validation_data=bag_of_words_val_ds,
    epochs=10,
    callbacks=[early_stopping, model_checkpoint],
)

In [ ]:
from utils import plot_history

In [ ]:
plot_history(history);

In [ ]:
bow_model = keras.models.load_model("models/bow_model.keras")
bow_acc = bow_model.evaluate(bag_of_words_test_ds)[1]
bow_acc

### Bi-Gram Model

#### Text Vectorization

In [ ]:
max_tokens = 30_000

text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="multi_hot",
    ngrams=2,
)

In [ ]:
text_vectorization.adapt(train_ds_no_labels)

In [ ]:
bigram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bigram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bigram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)

In [ ]:
x, y = next(bigram_train_ds.as_numpy_iterator())
x.shape

In [ ]:
text_vectorization.get_vocabulary()[100:108]

#### Model Building

In [ ]:
name = "bigram_classifier"

inputs = keras.Input(shape=(max_tokens,))
outputs = layers.Dense(1, activation="sigmoid")(inputs)
model = keras.Model(inputs, outputs, name=name)

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=2,
)
model_checkpoint = keras.callbacks.ModelCheckpoint(
    "models/bigram_model.keras", save_best_only=True
)

In [ ]:
history = model.fit(
    bigram_train_ds,
    validation_data=bigram_val_ds,
    epochs=10,
    callbacks=[early_stopping, model_checkpoint],
)

In [ ]:
plot_history(history);

In [ ]:
bigram_model = keras.models.load_model("models/bigram_model.keras")
bigram_acc = bigram_model.evaluate(bigram_test_ds)[1]
bigram_acc

## 3 Sequence Models

### LSTM

#### Text Vectorization

In [ ]:
max_length = 600
max_tokens = 30_000

text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="int",
    output_sequence_length=max_length,
)
text_vectorization.adapt(train_ds_no_labels)

In [ ]:
sequence_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
sequence_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
sequence_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)

In [ ]:
x, y = next(sequence_test_ds.as_numpy_iterator())
x.shape

#### One-Hot Encoder

In [ ]:
from keras import ops

class OneHotEncoding(keras.Layer):
    def __init__(self, depth, **kwargs):
        super().__init__(**kwargs)
        self.depth = depth

    def call(self, inputs):
        flat_inputs = ops.reshape(ops.cast(inputs, "int"), [-1])
        one_hot_vectors = ops.eye(self.depth)
        outputs = ops.take(one_hot_vectors, flat_inputs, axis=0)
        return ops.reshape(outputs, ops.shape(inputs) + (self.depth,))

one_hot_encoding = OneHotEncoding(max_tokens)

In [ ]:
x, y = next(sequence_train_ds.as_numpy_iterator())
one_hot_encoding(x).shape

#### Model Building

In [ ]:
hidden_dim = 64

inputs = keras.Input(shape=(max_length,), dtype="int32")
x = one_hot_encoding(inputs)
x = layers.Bidirectional(layers.LSTM(hidden_dim))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="lstm_with_one_hot")

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=2,
)
model_checkpoint = keras.callbacks.ModelCheckpoint(
    "models/lstm_model.keras", save_best_only=True
)

In [ ]:
model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=[early_stopping, model_checkpoint],
)

45 minutes per iteration * 10 = 7.5 hours

In [ ]:
lstm_model = keras.models.load_model("models/lstm_model.keras")
lstm_acc = lstm_model.evaluate(sequence_test_ds)[1]
lstm_acc

### LSTM with Word Embedding

In [ ]:
hidden_dim = 64

inputs = keras.Input(shape=(max_length,), dtype="int32")
x = keras.layers.Embedding(
    input_dim=max_tokens,
    output_dim=hidden_dim,
    mask_zero=True,
)(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(hidden_dim))(x)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="lstm_with_embedding")

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=2,
)
model_checkpoint = keras.callbacks.ModelCheckpoint(
    "models/lstm_embedding.keras", save_best_only=True
)

In [ ]:
model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=[early_stopping, model_checkpoint],
)

2 hours * 10 = 20 hours

In [ ]:
lstm_with_embedding_model = keras.models.load_model("models/lstm_with_embedding.keras")
lstm_with_embedding_acc = lstm_model.evaluate(sequence_test_ds)[1]
lstm_with_embedding_acc

### Pretraining Word Embedding

In [ ]:
imdb_vocabulary = text_vectorization.get_vocabulary()
tokenize_no_padding = keras.layers.TextVectorization(
    vocabulary=imdb_vocabulary,
    split="whitespace",
    output_mode="int",
)

#### Creating CBOW Dataset

In [ ]:
import tensorflow as tf

context_size = 4
window_size = 9

def window_data(token_ids):
    num_windows = tf.maximum(tf.size(token_ids) - context_size * 2, 0)
    windows = tf.range(window_size)[None, :]
    windows = windows + tf.range(num_windows)[:, None]
    windowed_tokens = tf.gather(token_ids, windows)

    return tf.data.Dataset.from_tensor_slices(windowed_tokens)

def split_label(window):
    left = window[:context_size]
    right = window[context_size + 1 :]
    bag = tf.concat((left, right), axis=0)
    label = window[4]

    return bag, label

dataset = keras.utils.text_dataset_from_directory(
    imdb_extract_dir / "train", batch_size=None
)
dataset = dataset.map(lambda x, y: x, num_parallel_calls=8)
dataset = dataset.map(tokenize_no_padding, num_parallel_calls=8)
dataset = dataset.interleave(window_data, cycle_length=8, num_parallel_calls=8)
dataset = dataset.map(split_label, num_parallel_calls=8)

#### CBOW Model Training

In [ ]:
hidden_dim = 64

inputs = keras.Input(shape=(2 * context_size,))
cbow_embedding = layers.Embedding(
    max_tokens,
    hidden_dim,
)
x = cbow_embedding(inputs)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(max_tokens, activation="sigmoid")(x)
cbow_model = keras.Model(inputs, outputs)

In [ ]:
cbow_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)

In [ ]:
model_checkpoint = keras.callbacks.ModelCheckpoint(
    "models/cbow_model.keras", save_best_only=True
)

In [ ]:
dataset = dataset.batch(1024).cache()
cbow_model.fit(dataset, epochs=4, callbacks=[model_checkpoint])

#### Loading Pretrained Embedding for Classification

In [ ]:
inputs = keras.Input(shape=(max_length,))
lstm_embedding = layers.Embedding(
    input_dim=max_tokens,
    output_dim=hidden_dim,
    mask_zero=True,
)
x = lstm_embedding(inputs)
x = layers.Bidirectional(layers.LSTM(hidden_dim))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="lstm_with_cbow")

In [ ]:
cbow_model = keras.models.load_model("models/cbow_model.keras")
cbow_embedding = cbow_model.layers[1]

In [ ]:
lstm_embedding.embeddings.assign(cbow_embedding.embeddings)

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=2,
)
model_checkpoint = keras.callbacks.ModelCheckpoint(
    "models/lstm_pretrained_embedding.keras", save_best_only=True
)

In [ ]:
model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=[early_stopping, model_checkpoint],
)

2 hours * 10 = 20 hours

In [ ]:
lstm_with_pretrained_embedding = keras.models.load_model(
    "models/lstm_pretrained_embedding.keras")
lstm_pretrained_embedding_acc = lstm_with_pretrained_embedding.evaluate(sequence_test_ds)[1]
lstm_pretrained_embedding_acc